# Mô Phỏng Máy Hút Bụi (Vacuum Cleaner Agent)

**Quy ước ma trận:**
- `0` → Ô trống (sạch) / Máy hút bụi
- `1` → Bụi

**Thuật toán:** Forward Checking — nhìn trước N bước để đánh giá hướng đi tối ưu, ưu tiên đường có bụi.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

In [ ]:
# ── Cấu hình ──
ROWS      = 5
COLS      = 7
DUST_PROB = 0.4

# ── Tạo môi trường ──
def create_env(rows, cols, dust_prob):
    grid = np.zeros((rows, cols), dtype=int)
    for r in range(rows):
        for c in range(cols):
            if random.random() < dust_prob:
                grid[r][c] = 1
    return grid

# ── Vẽ ma trận ──
def draw_grid(grid, pos, title):
    rows, cols = grid.shape
    fig, ax = plt.subplots(figsize=(cols * 0.9, rows * 0.9))

    cmap = ListedColormap(['#F0F0F0', '#F4D03F'])  # 0=xám nhạt, 1=vàng
    ax.imshow(grid, cmap=cmap, vmin=0, vmax=1)

    for r in range(rows):
        for c in range(cols):
            if (r, c) == pos:
                ax.text(c, r, '🤖', ha='center', va='center', fontsize=14)
            elif grid[r][c] == 1:
                ax.text(c, r, '●', ha='center', va='center',
                        fontsize=16, color='#884400')
            else:
                ax.text(c, r, '0', ha='center', va='center',
                        fontsize=11, color='#888888')

    ax.set_xticks(np.arange(cols))
    ax.set_yticks(np.arange(rows))
    ax.set_xticklabels(np.arange(cols))
    ax.set_yticklabels(np.arange(rows))
    ax.set_xticks(np.arange(-0.5, cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, rows, 1), minor=True)
    ax.grid(which='minor', color='#AAAAAA', linewidth=0.8)
    ax.tick_params(which='minor', bottom=False, left=False)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)

    legend = [
        mpatches.Patch(color='#F0F0F0', label='0 - Ô sạch / Máy'),
        mpatches.Patch(color='#F4D03F', label='1 - Bụi'),
    ]
    ax.legend(handles=legend, loc='upper right',
              bbox_to_anchor=(1.35, 1.02), fontsize=9)

    plt.tight_layout()
    plt.show()


# ── Khởi tạo ──
grid = create_env(ROWS, COLS, DUST_PROB)
total_dust = int(np.sum(grid == 1))

print(f"Ma trận ban đầu  |  Tổng bụi: {total_dust} ô")
draw_grid(grid, pos=(-1, -1), title=f'Ma trận ban đầu — Bụi: {total_dust} ô')

In [ ]:
# ── Agent - Forward Checking (Nhìn trước + Đánh giá) ──
def run_agent(grid_in):
    grid = grid_in.copy()
    rows, cols = grid.shape
    steps = 0
    cleaned = 0
    LOOK_AHEAD = 3  # Tầm nhìn trước (số bước)

    visited = set()
    path_stack = []

    directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]
    dir_names = {(0, 1): 'PHẢI', (1, 0): 'XUỐNG', (0, -1): 'TRÁI', (-1, 0): 'LÊN'}

    # Hàm forward checking: BFS nhìn trước look_ahead bước, đếm bụi
    def forward_check(sr, sc, visited_set, look_ahead=LOOK_AHEAD):
        """Nhìn trước look_ahead bước từ ô (sr,sc), đếm số bụi chưa được hút."""
        dust_count = 0
        frontier = [(sr, sc, 0)]  # (row, col, depth)
        checked = visited_set | {(sr, sc)}

        while frontier:
            cr, cc, depth = frontier.pop(0)
            if depth >= look_ahead:
                continue
            for dr, dc in directions:
                nr, nc = cr + dr, cc + dc
                if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in checked:
                    checked.add((nr, nc))
                    if grid[nr][nc] == 1:  # Còn bụi trong tầm nhìn
                        dust_count += 1
                    frontier.append((nr, nc, depth + 1))

        return dust_count

    # Bắt đầu tại (0, 0)
    r, c = 0, 0
    visited.add((r, c))
    path_stack.append((r, c))

    print(f'Bắt đầu tại ({r}, {c})')
    if grid[r][c] == 1:
        grid[r][c] = 0
        cleaned += 1
        print(f'   ⟹  Phát hiện bụi! Hút bụi tại ({r}, {c}) — Đã hút: {cleaned}/{total_dust}')
    else:
        print(f'   ⟹  Ô sạch, tiếp tục.')
    draw_grid(grid, pos=(r, c),
              title=f'Bắt đầu | Vị trí: ({r},{c}) | Đã hút: {cleaned}/{total_dust}')

    while len(visited) < rows * cols:
        # Forward checking: đánh giá tất cả hướng đi khả dụng
        candidates = []
        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                # Nhìn trước để đếm bụi trong phạm vi LOOK_AHEAD bước
                future_dust = forward_check(nr, nc, visited | {(nr, nc)})
                # Điểm ưu tiên: có bụi ngay tại ô → +1000; mỗi bụi trong tầm nhìn → +1
                immediate = 1000 if grid[nr][nc] == 1 else 0
                score = immediate + future_dust
                candidates.append((score, nr, nc, dr, dc, future_dust))

        if candidates:
            # Sắp xếp theo điểm giảm dần → chọn hướng tốt nhất
            candidates.sort(key=lambda x: x[0], reverse=True)
            best_score, nr, nc, dr, dc, future_dust = candidates[0]

            steps += 1
            direction = dir_names[(dr, dc)]

            # Hiển thị thông tin forward checking
            fc_parts = []
            for score, mr, mc, _, _, fd in candidates:
                marker = '→' if (mr, mc) == (nr, nc) else ' '
                fc_parts.append(f'{marker}({mr},{mc}):điểm={score}(bụi_gần={fd})')
            fc_info = ' | '.join(fc_parts)
            print(f'  [Forward Check] {fc_info}')

            move_msg = f'Bước {steps}: Di chuyển {direction} → ô ({nr}, {nc})'

            r, c = nr, nc
            visited.add((r, c))
            path_stack.append((r, c))

            print(move_msg)

            if grid[r][c] == 1:
                grid[r][c] = 0
                cleaned += 1
                action_msg = f'   ⟹  Phát hiện bụi! Hút bụi tại ({r}, {c}) — Đã hút: {cleaned}/{total_dust}'
            else:
                action_msg = f'   ⟹  Ô sạch (forward check thấy {future_dust} bụi trong tầm nhìn), tiếp tục.'

            print(action_msg)
            draw_grid(grid, pos=(r, c),
                      title=f'Bước {steps} | Vị trí: ({r},{c}) | FC | Đã hút: {cleaned}/{total_dust}')
        else:
            # Không còn ô chưa thăm kề cận → quay lui
            if len(path_stack) > 1:
                path_stack.pop()
                pr, pc = path_stack[-1]

                steps += 1
                if r > pr:
                    direction = 'LÊN (quay lui)'
                elif r < pr:
                    direction = 'XUỐNG (quay lui)'
                elif c > pc:
                    direction = 'TRÁI (quay lui)'
                else:
                    direction = 'PHẢI (quay lui)'

                print(f'Bước {steps}: Quay lui {direction} → ô ({pr}, {pc})')
                print(f'   ⟹  Forward check: không còn hướng đi khả dụng, quay lui về nút cha.')

                r, c = pr, pc
                draw_grid(grid, pos=(r, c),
                          title=f'Bước {steps} | Vị trí: ({r},{c}) | Quay lui | Đã hút: {cleaned}/{total_dust}')
            else:
                break

    # Kết quả
    print('=' * 45)
    if cleaned == total_dust:
        status = 'THÀNH CÔNG'
        reason = 'Forward checking đã dẫn đường đến tất cả bụi, hút sạch hết'
    else:
        status = 'THẤT BẠI'
        reason = 'Không hút hết bụi'

    print(f'Số bước đi  : {steps}')
    print(f'Bụi đã hút  : {cleaned} / {total_dust} ô')
    print(f'Trạng thái  : {status}')
    print(f'Lý do       : {reason}')


run_agent(grid)
